# Build 03-07 · Retuning — recall pinned to production, how much precision comes back

**Kernel: the analysis `.venv`** (`python3`) — reads parquet + CSV only, loads no model.

03_05 tunes every axis to hold precision ≥ `config.TARGET_PRECISION` (0.985). That target is
confirmed (supervisor, 2026-09-28) to be **v3's own requirement, never v2's** — v2 was live at
whatever precision its real deployed tau actually gave, not at 0.985. Forcing 0.985 collapses
recall to near zero for v3, and 03_05's own `tuned_holdout` numbers show v2 cannot reach 0.985
out-of-sample at ANY recall (real OOT-garage precision tops out around 0.97-0.98, never 0.985).
Neither number answers the question this notebook asks instead:

> given that production already runs at some recall (whatever the real deployed tau delivers),
> how much precision can a corrected model buy while not giving up that recall?

Mechanically: `threshold.tune(..., recall_min=anchor)` instead of the default precision-floor
mode, run on the SAME clean holdout slice 03_05 uses for `tuned_holdout` (a split neither the
fit nor the final eval slice touches), then measured on the eval slice — same discipline, only
the selection criterion changes.

The recall floor itself is **never hardcoded** (see memory: no raw Allianz numbers in code) —
§2b below reads it off baseline's own real recall at its own real deployed tau(s), live, every
run.

### 실행 전 설정

**커널**: analysis `.venv` (`python3`)

**바꿀 것 (§1)**: `VERSION` — `"v2"` 또는 `"v3"` (`HOLDOUT_SPLIT`은 여기서 자동으로 정해짐: v2는 `val`, v3는 `test`)

**선행 조건**: `03_04_prediction.ipynb`까지 다 돌려서 각 axis의 mitigated score가 `reeval/<v>/`에 있어야 함. `03_05_reevaluation.ipynb`은 선행 필요 없음 (이 노트북은 매번 원시 점수에서 다시 집계함, 03_05의 출력을 읽지 않음).

**fixed-tau 변이는 모두 제외된다**: `rarity_fixed`/`pnu_fixed`는 논문 본문에서 빼기로 해서 (fixed vs regime tau 구별은 결과가 아니라 구현 디테일이다), 이 노트북의 `MODELS` 리스트에서도 같이 뺀다.

In [ ]:
# §0 — setup (analysis .venv kernel)
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_curve, roc_auc_score

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import config
import figstyle
import threshold

figstyle.apply()
pd.set_option("display.width", 160)
print("ROOT =", ROOT)

In [ ]:
# §1 — RUN_SPEC
VERSION = "v3"        # "v2" works too
TRAIN_SPLIT = "train"                 # unused here (03_05's "tuned"/train-tuned axis is not
                                       # repeated in this notebook — only the honest holdout
                                       # discipline is), kept for symmetry with 03_05's RUN_SPEC
EVAL_SPLIT = config.OOT_SPLIT[VERSION]            # v1/v2 invert "test" — never picked by name
HOLDOUT_SPLIT = {"v2": "val", "v3": "test"}[VERSION]   # a split neither the fit nor EVAL_SPLIT
                                                          # touches — same slice 03_05 uses for
                                                          # tuned_holdout
ID_COL = "claim_id"
SCORE_COL = "model_" + VERSION + "_score"

_rule = config.DECISION_RULES[VERSION]
if _rule["shape"] == "global":
    TAU_GRID = [float(_rule["threshold"])]
elif _rule["shape"] == "piecewise_global":
    TAU_GRID = sorted({float(r["threshold"]) for r in _rule["regimes"]})
else:
    raise SystemExit("segmented rule (v1) is out of scope here")

# v2's regime[0] (0.8915) is not a confirmed documented tau — config.py's own DECISION_RULES['v2']
# comment says "nothing may read regimes[0] as the documented tau". The other two grid values
# (0.872, 0.825) are supervisor-confirmed real production cutoffs — keep only those, so every
# anchor built below sits on a tau that genuinely ran in production.
if VERSION == "v2":
    TAU_GRID = [t for t in TAU_GRID if abs(t - 0.8915) > 1e-9]

REEVAL_DIR = ROOT / "src" / "data" / "real" / "reeval" / VERSION
figstyle.FIG_DIR = ROOT / "figures" / "mitigation" / "03_07" / VERSION
figstyle.ALIAS_SPLIT = False    # see 03_05 §1's note — this notebook never touches feature names
print("eval split:", EVAL_SPLIT, "| holdout split:", HOLDOUT_SPLIT, "| production tau grid:", TAU_GRID)
print("FIG_DIR =", figstyle.FIG_DIR)

In [ ]:
# §2 — assemble: corrector_targets (treatment pre-joined by 03_01) + one score column per model.
# Same shape as 03_05 §2 — kept self-contained rather than imported, matching this repo's
# per-notebook convention (each Build 03 notebook stands alone).
def load_split_frame(split: str) -> pd.DataFrame:
    """corrector_targets of `split` — targets already joined to the recorded treatment (03_01)."""
    p = config.split_path("corrector_targets", VERSION, split)
    assert p.is_file(), f"{p} missing — run 03_01_corrector_inputs first"
    m = pd.read_parquet(p)
    m["date"] = pd.to_datetime(m["date"])
    print(f"{VERSION} {split}: {len(m):,} treated rows (unmatched already dropped in 03_01)")
    return m


def attach_scores(frame: pd.DataFrame, split: str) -> tuple[pd.DataFrame, list[str]]:
    """Merge every available model's scores on `split`: mitigated axes + baseline if exported."""
    models = []
    bpath = config.path("scores", VERSION, split=split)
    if bpath.is_file():
        b = pd.read_parquet(bpath)[[ID_COL, SCORE_COL]].rename(columns={SCORE_COL: "baseline"})
        frame = frame.merge(b, on=ID_COL, how="left")
        models.append("baseline")
    else:
        print(f"  note: no baseline scores at {bpath} (run src/scoring/predict.py for {VERSION}) — skipped")
    prefix = f"{VERSION}_mitigated_scores_{split}_"
    for p in sorted(REEVAL_DIR.glob(prefix + "*.parquet")):
        tag = p.stem[len(prefix):]
        s = pd.read_parquet(p)[[ID_COL, SCORE_COL]].rename(columns={SCORE_COL: tag})
        frame = frame.merge(s, on=ID_COL, how="left")
        models.append(tag)
    return frame, models


def drop_fixed(models: list[str]) -> list[str]:
    """Paper scope excludes the fixed-tau rarity/pnu variants — see the setup note above."""
    return [m for m in models if not m.endswith("_fixed")]


ev, MODELS = attach_scores(load_split_frame(EVAL_SPLIT), EVAL_SPLIT)
hd, HD_MODELS = attach_scores(load_split_frame(HOLDOUT_SPLIT), HOLDOUT_SPLIT)
MODELS, HD_MODELS = drop_fixed(MODELS), drop_fixed(HD_MODELS)
assert MODELS, "no score files for the eval split — run 03_04 first"

ev_matched = ev["decision"].notna()
ev_gar = ev.loc[ev_matched & (ev["decision"] == 0)]
hd_matched = hd["decision"].notna()
hd_gar = hd.loc[hd_matched & (hd["decision"] == 0)]
print(f"eval garage rows: {len(ev_gar):,} | holdout ({HOLDOUT_SPLIT}) garage rows: {len(hd_gar):,}")
print("models:", MODELS)


def load_full_split(split: str) -> tuple[pd.DataFrame, list[str]]:
    """targets(split) — every row with its recorded label — plus one score column per model."""
    t = pd.read_parquet(config.split_path("targets", VERSION, split))[[ID_COL, "observed"]]
    frame, models = attach_scores(t, split)
    return frame, drop_fixed(models)


ev_all, all_models = load_full_split(EVAL_SPLIT)
assert all_models == MODELS, "score files differ between the treated and the full read — rerun §2"


def metrics_at(y: np.ndarray, s: np.ndarray, tau: float) -> dict:
    """AUC + precision/recall/CM at the STRICT rule score > tau (score == tau garages)."""
    pred = s > tau
    tp = int((pred & (y == 1)).sum()); fp = int((pred & (y == 0)).sum())
    tn = int((~pred & (y == 0)).sum()); fn = int((~pred & (y == 1)).sum())
    return {"AUC": float(roc_auc_score(y, s)),
            "precision": tp / (tp + fp) if tp + fp else float("nan"),
            "recall": tp / (tp + fn) if tp + fn else float("nan"),
            "TN": tn, "FP": fp, "FN": fn, "TP": tp}

### §2b. recall-min 값을 구하는 방식

baseline이 실제로 배포된 tau(`TAU_GRID`, §1)에서 eval garage(검증된 로우)로 잰 recall을 그대로 읽어서, 그 값을 "recall은 이 밑으로 떨어지면 안 된다"는 floor로 쓴다. 값은 매번 코드가 직접 계산해서 출력만 하고, 숫자를 코드에 박아넣지는 않는다.

In [ ]:
# §2b — the recall floor itself: baseline's OWN real recall, on the eval garage slice, at its
# OWN real deployed tau (TAU_GRID). Read live every run — never hardcoded. One anchor per grid
# tau (v2: two eras' cutoffs; v3: one).
if "baseline" not in MODELS:
    raise SystemExit(f"§2b needs baseline scores for {VERSION} — run src/scoring/predict.py first")

g0 = ev_gar[[ID_COL, "observed", "baseline"]].dropna(subset=["baseline"])
y0 = g0["observed"].astype(int).to_numpy()
s0 = g0["baseline"].to_numpy(dtype=float)

anchor_rows = []
for tau in TAU_GRID:
    m = metrics_at(y0, s0, tau)
    anchor_rows.append({"anchor_tau": tau, "baseline_precision_garage": round(m["precision"], 4),
                         "baseline_recall_garage": round(m["recall"], 4), "n": len(g0)})
anchors = pd.DataFrame(anchor_rows)
display(anchors)
figstyle.save_table(anchors, f"{VERSION}_0307_sec2b_recall_min_anchors_{EVAL_SPLIT}", index=False)

### §3. 같은 recall을 지키면 precision이 얼마나 돌아오는지

`§2b`의 anchor마다, `§1`의 `HOLDOUT_SPLIT`(fit도 eval도 안 본 진짜 깨끗한 슬라이스, 03_05의 `tuned_holdout`이 쓰는 데이터와 동일)에서 `threshold.tune(..., recall_min=anchor)`로 tau를 고른다. 선택은 holdout에서만 하고, 측정은 eval garage/whole 둘 다에서 한다 — 03_05의 구조와 동일하게 선택과 측정 슬라이스만 서로 갈린다.

In [ ]:
# §3 — for every (model, anchor), the highest holdout tau whose holdout recall still clears the
# anchor (threshold.tune(..., recall_min=...), STRICT >, see threshold.py's own note that this
# mode's boundary is inherited "as received" from Allianz's own select_best_threshold) — same
# discipline as 03_05's tuned_holdout: chosen on a split the model's fit and the final eval slice
# both never touch, then MEASURED on the eval slice, never on the slice it was chosen on.
rows = []
for name in MODELS:
    if name not in HD_MODELS:
        print(f"  {name}: no scores on {HOLDOUT_SPLIT} — skipped")
        continue
    hg = hd_gar[["observed", name]].dropna(subset=[name])
    y_hd = hg["observed"].astype(int).to_numpy()
    s_hd = hg[name].to_numpy(dtype=float)

    g = ev_gar[[ID_COL, "observed", name]].dropna(subset=[name])
    y_gar = g["observed"].astype(int).to_numpy()
    s_gar = g[name].to_numpy(dtype=float)

    a = ev_all[[ID_COL, "observed", name]].dropna(subset=[name])
    y_all = a["observed"].astype(int).to_numpy()
    s_all = a[name].to_numpy(dtype=float)
    s_all_eval = ev[[ID_COL, name]].dropna(subset=[name])   # every scored row — for the decisions file

    for _, arow in anchors.iterrows():
        anchor_tau, anchor_recall = float(arow["anchor_tau"]), float(arow["baseline_recall_garage"])
        t = threshold.tune(y_hd, s_hd, recall_min=anchor_recall)
        if t is None:
            print(f"  {name}: holdout recall never exceeds the anchor "
                  f"({anchor_recall:.4f}, from production tau={anchor_tau}) — no cutoff picked")
            rows.append({"model": name, "anchor_tau": anchor_tau, "anchor_recall": anchor_recall,
                         "tau": None, "reached": False})
            continue

        gar_m = metrics_at(y_gar, s_gar, float(t))
        all_m = metrics_at(y_all, s_all, float(t))
        rows.append({"model": name, "anchor_tau": anchor_tau, "anchor_recall": anchor_recall,
                     "tau": round(float(t), 6), "reached": True,
                     "precision_garage": round(gar_m["precision"], 4),
                     "recall_garage": round(gar_m["recall"], 4),
                     "precision_whole": round(all_m["precision"], 4),
                     "recall_whole": round(all_m["recall"], 4),
                     "TP_garage": gar_m["TP"], "FP_garage": gar_m["FP"],
                     "FN_garage": gar_m["FN"], "TN_garage": gar_m["TN"]})

        dp = REEVAL_DIR / f"{VERSION}_decisions_{EVAL_SPLIT}_{name}_recallmin{round(anchor_recall, 4)}_tau{round(float(t), 6)}.parquet"
        pd.DataFrame({ID_COL: s_all_eval[ID_COL].to_numpy(),
                      "score": s_all_eval[name].to_numpy(dtype=float),
                      "decision": (s_all_eval[name].to_numpy(dtype=float) > float(t)).astype(int)}
                     ).to_parquet(dp, index=False)

retune = pd.DataFrame(rows)
display(retune)
figstyle.save_table(retune, f"{VERSION}_0307_sec3_retune_metrics_{EVAL_SPLIT}", index=False)

### §4. PR curve 위에 생산 recall 위치 표시

점선은 03_05의 0.985 target (subsec B), 수직선은 §2b의 anchor recall(들)이다. 이 둘이 곡선을 어디서 침범하는지가 핵심이다.

In [ ]:
fig, ax = plt.subplots(figsize=figstyle.FIG_1)
for name in MODELS:
    g = ev_gar[["observed", name]].dropna(subset=[name])
    y = g["observed"].astype(int).to_numpy()
    prec, rec, _ = precision_recall_curve(y, g[name].to_numpy(dtype=float))
    if name == "baseline":
        ax.plot(rec[:-1], prec[:-1], ls=":", color=figstyle.NEUTRAL, label="baseline")
    else:
        ax.plot(rec[:-1], prec[:-1], label=name)
ax.axhline(config.TARGET_PRECISION, ls="--", lw=1, color=figstyle.INK)
ax.text(0.01, config.TARGET_PRECISION + 0.01, f"{config.TARGET_PRECISION} target (03_05, subsec B)",
        fontsize=8, color=figstyle.INK)
for _, arow in anchors.iterrows():
    ax.axvline(float(arow["baseline_recall_garage"]), ls=":", lw=1, color=figstyle.NEUTRAL)
ax.set_xlabel("recall"); ax.set_ylabel("precision"); ax.set_ylim(0, 1.02)
ax.set_title(f"{VERSION} — PR on {EVAL_SPLIT} garage, production recall anchors marked")
ax.legend(fontsize=8)
figstyle.save_fig(fig, f"{VERSION}_0307_sec4_retune_pr_{EVAL_SPLIT}")
plt.show()

### §4b. 같은 그림, whole split(강제 라벨 포함)

§4와 같은 anchor·target 선을 쓰지만, 곡선은 `ev_all`(garage + 강제 스크랩 + 미처리) 위에서 그린다. 두 그림의 세로 차이가 03_05 §4/§4b가 보여준 forced-label precision 왜곡과 같은 것이다.

In [ ]:
fig, ax = plt.subplots(figsize=figstyle.FIG_1)
for name in MODELS:
    a = ev_all[["observed", name]].dropna(subset=[name])
    y = a["observed"].astype(int).to_numpy()
    prec, rec, _ = precision_recall_curve(y, a[name].to_numpy(dtype=float))
    if name == "baseline":
        ax.plot(rec[:-1], prec[:-1], ls=":", color=figstyle.NEUTRAL, label="baseline")
    else:
        ax.plot(rec[:-1], prec[:-1], label=name)
ax.axhline(config.TARGET_PRECISION, ls="--", lw=1, color=figstyle.INK)
ax.text(0.01, config.TARGET_PRECISION + 0.01, f"{config.TARGET_PRECISION} target (03_05, subsec B)",
        fontsize=8, color=figstyle.INK)
for _, arow in anchors.iterrows():
    ax.axvline(float(arow["baseline_recall_garage"]), ls=":", lw=1, color=figstyle.NEUTRAL)
ax.set_xlabel("recall"); ax.set_ylabel("precision"); ax.set_ylim(0, 1.02)
ax.set_title(f"{VERSION} — PR on {EVAL_SPLIT}, whole split (forced labels in), anchors marked")
ax.legend(fontsize=8)
figstyle.save_fig(fig, f"{VERSION}_0307_sec4b_retune_pr_{EVAL_SPLIT}_all")
plt.show()